# Single modality base line model: this time elastic net 

### Preparation

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_validate, GroupShuffleSplit, GroupKFold

In [3]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl")
df.head()

,SequencingID,ModelID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),FKBP4 (2288),...,Bit_1014,Bit_1015,Bit_1016,Bit_1017,Bit_1018,Bit_1019,Bit_1020,Bit_1021,Bit_1022,Bit_1023
0,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
1,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0
2,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,0,0,0,0,0,0
3,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,0,1,1,0,0,0,0
4,CDS-0CqPRM,ACH-000242,7.10594,2.700053,5.803904,5.66261,4.231824,2.283468,4.735717,6.484923,...,0,0,0,1,0,1,0,0,0,0


In [8]:
def genomic_baseline_model(df, target):
    # Prepare data
    X_genomic = df.iloc[:, 2:980].values # L1000 landmark genes
    y_genomic = df[target].values # y: drug response

    target_size = 50000 / len(df)

    # ensures that all rows from one cell line stay together in the train/test split
    gss = GroupShuffleSplit(n_splits=1, train_size=target_size, random_state=42)

    train_idx, _ = next(gss.split(df, groups=df['ModelID']))
    df_small = df.iloc[train_idx].copy()

    X_small = df_small.iloc[:, 2:980].values.astype('float32')
    y_small = df_small[target].values.astype('float32')
    groups_small = df_small['ModelID'].values

    print(f"Number of unique cell lines: {len(np.unique(groups_small))}")

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNetCV(
            l1_ratio=[.1, .5, .7, .9, .95, .99, 1], # model will find the best mix
            cv=5, 
            random_state=42,
            max_iter=5000,
            n_jobs=1 # Keeping it to 1 to avoid memory issues
        ))
    ])

    print("Starting Elastic Net Cross-Validation...")
    cv_results = cross_validate(
        pipeline, 
        X_small, 
        y_small, 
        groups=groups_small, 
        cv=GroupKFold(n_splits=5),
        scoring=['r2', 'neg_mean_squared_error'],
        return_estimator=True,
        n_jobs=1
    )

    # Calculate Results
    avg_r2 = np.mean(cv_results['test_r2'])
    avg_rmse = np.sqrt(-np.mean(cv_results['test_neg_mean_squared_error']))

    print(f"\n--- Elastic Net Results ---")
    print(f"Average R² Score: {avg_r2:.4f}")
    print(f"Average RMSE:     {avg_rmse:.4f}")

    best_model = cv_results['estimator'][0].named_steps['model']
    coefs = best_model.coef_

    # Get the gene names from your original columns
    gene_names = df_small.iloc[:, 2:980].columns

    # Create a summary table
    features_df = pd.DataFrame({'Gene': gene_names, 'Coefficient': coefs})
    features_df['Abs_Coef'] = features_df['Coefficient'].abs()

    # Filter for genes the model didn't set to zero
    selected_genes = features_df[features_df['Coefficient'] != 0]

    print(f"\nElastic Net selected {len(selected_genes)} genes out of 978.")
    print(f"Top 5 Positive Biomarkers (Increase {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
    print(f"\nTop 5 Negative Biomarkers (Decrease {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=True).head(5))
        

In [9]:
def chemical_baseline_model(df, target):
    target_size = 50000 / len(df)

    # ensures that all rows from one cell line stay together in the train/test split
    gss = GroupShuffleSplit(n_splits=1, train_size=target_size, random_state=42)

    train_idx, _ = next(gss.split(df, groups=df['ModelID']))
    df_small = df.iloc[train_idx].copy()

    X_small = df_small.loc[:, df_small.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].values.astype(float)
    y_small = df_small[target].values.astype('float32')
    groups_small = df_small['DRUG_ID'].values

    print(f"Number of unique drugs: {len(np.unique(groups_small))}")

    # Use StandardScaler to scale the features and ElasticNetCV for regression with cross-validation
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNetCV(
            l1_ratio=[.1, .5, .7, .9, .95, .99, 1], # model will find the best mix
            cv=5, 
            random_state=42,
            max_iter=5000,
            n_jobs=1 # Keeping it to 1 to avoid memory issues
        ))
    ])

    print("Starting Elastic Net Cross-Validation...")
    cv_results = cross_validate(
        pipeline, 
        X_small, 
        y_small, 
        groups=groups_small, 
        cv=GroupKFold(n_splits=5),
        scoring=['r2', 'neg_mean_squared_error'],
        return_estimator=True,
        n_jobs=1
    )

    # calculate Results
    avg_r2 = np.mean(cv_results['test_r2'])
    avg_rmse = np.sqrt(-np.mean(cv_results['test_neg_mean_squared_error']))

    print(f"\n--- Elastic Net Results ---")
    print(f"Average R² Score: {avg_r2:.4f}")
    print(f"Average RMSE:     {avg_rmse:.4f}")

    best_model = cv_results['estimator'][0].named_steps['model']
    coefs = best_model.coef_

    # Get the molecule structure names from your original columns
    bit_names = df_small.loc[:, df_small.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].columns

    # Create a summary table
    features_df = pd.DataFrame({'Molecule_Structure': bit_names, 'Coefficient': coefs})
    features_df['Abs_Coef'] = features_df['Coefficient'].abs()

    # Filter for genes the model didn't set to zero
    selected_struct = features_df[features_df['Coefficient'] != 0]

    print(f"\nElastic Net selected {len(selected_struct)} molecule structures out of 1032.")
    print(f"Top 5 Positive molecule structures (Increase {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
    print(f"\nTop 5 Negative molecule structures (Decrease {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

## AUC

### Lol first try with genomic data only and AUC as target value

Define training set

In [5]:
# training set for genomic features
X_genomic = df.iloc[:, 2:980].values # L1000 landmark genes
y_genomic = df['AUC'].values # y: drug response

In [7]:
target_size = 50000 / len(df)

# ensures that all rows from one cell line stay together in the train/test split
gss = GroupShuffleSplit(n_splits=1, train_size=target_size, random_state=42)

train_idx, _ = next(gss.split(df, groups=df['ModelID']))
df_small = df.iloc[train_idx].copy()

X_small = df_small.iloc[:, 2:980].values.astype('float32')
y_small = df_small['AUC'].values.astype('float32')
groups_small = df_small['ModelID'].values

print(f"Number of unique cell lines: {len(np.unique(groups_small))}")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1], # model will find the best mix
        cv=5, 
        random_state=42,
        max_iter=5000,
        n_jobs=1 # Keeping it to 1 to avoid memory issues
    ))
])

print("Starting Elastic Net Cross-Validation...")
cv_results = cross_validate(
    pipeline, 
    X_small, 
    y_small, 
    groups=groups_small, 
    cv=GroupKFold(n_splits=5),
    scoring=['r2', 'neg_mean_squared_error'],
    return_estimator=True,
    n_jobs=1
)

# 3. Calculate Results
avg_r2 = np.mean(cv_results['test_r2'])
avg_rmse = np.sqrt(-np.mean(cv_results['test_neg_mean_squared_error']))

print(f"\n--- Elastic Net Results ---")
print(f"Average R² Score: {avg_r2:.4f}")
print(f"Average RMSE:     {avg_rmse:.4f}")

best_model = cv_results['estimator'][0].named_steps['model']
coefs = best_model.coef_

# Get the gene names from your original columns
gene_names = df_small.iloc[:, 2:980].columns

# Create a summary table
features_df = pd.DataFrame({'Gene': gene_names, 'Coefficient': coefs})
features_df['Abs_Coef'] = features_df['Coefficient'].abs()

# Filter for genes the model didn't set to zero
selected_genes = features_df[features_df['Coefficient'] != 0]

print(f"\nElastic Net selected {len(selected_genes)} genes out of 978.")
print("Top 5 Positive Biomarkers (Increase AUC/Resistance):")
print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
print("\nTop 5 Negative Biomarkers (Decrease AUC/Sensitivity):")
print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

Number of unique cell lines: 191
Starting Elastic Net Cross-Validation...


c:\Users\Juli\anaconda3\envs\my-rdkit-env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.900e-02, tolerance: 6.560e-02
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Juli\anaconda3\envs\my-rdkit-env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.984e-01, tolerance: 6.560e-02
  model = cd_fast.enet_coordinate_descent_gram(



--- Elastic Net Results ---
Average R² Score: 0.0141
Average RMSE:     0.1436

Elastic Net selected 36 genes out of 978.
Top 5 Positive Biomarkers (Increase AUC/Resistance):
              Gene  Coefficient  Abs_Coef
887    MUC1 (4582)     0.007181  0.007181
284    RHEB (6009)     0.005292  0.005292
934  KIFBP (26128)     0.004856  0.004856
903   PTPN1 (5770)     0.002304  0.002304
559  TFAP2A (7020)     0.001845  0.001845

Top 5 Negative Biomarkers (Decrease AUC/Sensitivity):
               Gene  Coefficient  Abs_Coef
617  SCCPDH (51097)    -0.003767  0.003767
244      CSK (1445)    -0.003411  0.003411
554     FPGS (2356)    -0.002574  0.002574
335    PTPN6 (5777)    -0.002536  0.002536
105  CSNK2A2 (1459)    -0.002334  0.002334


### Elastic net with chemical data only

In [ ]:
target_size = 50000 / len(df)

# ensures that all rows from one cell line stay together in the train/test split
gss = GroupShuffleSplit(n_splits=1, train_size=target_size, random_state=42)

train_idx, _ = next(gss.split(df, groups=df['ModelID']))
df_small = df.iloc[train_idx].copy()

X_small = df_small.loc[:, df_small.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].values.astype(float)
y_small = df_small['AUC'].values.astype('float32')
groups_small = df_small['DRUG_ID'].values

print(f"Number of unique drugs: {len(np.unique(groups_small))}")

# Use StandardScaler to scale the features and ElasticNetCV for regression with cross-validation

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1], # model will find the best mix
        cv=5, 
        random_state=42,
        max_iter=5000,
        n_jobs=1 # Keeping it to 1 to avoid memory issues
    ))
])

print("Starting Elastic Net Cross-Validation...")
cv_results = cross_validate(
    pipeline, 
    X_small, 
    y_small, 
    groups=groups_small, 
    cv=GroupKFold(n_splits=5),
    scoring=['r2', 'neg_mean_squared_error'],
    return_estimator=True,
    n_jobs=1
)

# calculate Results
avg_r2 = np.mean(cv_results['test_r2'])
avg_rmse = np.sqrt(-np.mean(cv_results['test_neg_mean_squared_error']))

print(f"\n--- Elastic Net Results ---")
print(f"Average R² Score: {avg_r2:.4f}")
print(f"Average RMSE:     {avg_rmse:.4f}")

best_model = cv_results['estimator'][0].named_steps['model']
coefs = best_model.coef_

# Get the molecule structure names from your original columns
bit_names = df_small.loc[:, df_small.columns.str.startswith(('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_'))].columns

# Create a summary table
features_df = pd.DataFrame({'Molecule_Structure': bit_names, 'Coefficient': coefs})
features_df['Abs_Coef'] = features_df['Coefficient'].abs()

# Filter for genes the model didn't set to zero
selected_struct = features_df[features_df['Coefficient'] != 0]

print(f"\nElastic Net selected {len(selected_struct)} molecule structures out of 1032.")
print("Top 5 Positive molecule structures (Increase AUC/Resistance):")
print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
print("\nTop 5 Negative molecule structures (Decrease AUC/Sensitivity):")
print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

Number of unique drugs: 295
Starting Elastic Net Cross-Validation...

--- Elastic Net Results ---
Average R² Score: -0.4646
Average RMSE:     0.1714

Elastic Net selected 212 molecule structures out of 978.
Top 5 Positive molecule structures (Increase AUC/Resistance):
    Molecule_Structure  Coefficient  Abs_Coef
29              Bit_29     0.009600  0.009600
90              Bit_90     0.008979  0.008979
533            Bit_533     0.008174  0.008174
898            Bit_898     0.008118  0.008118
781            Bit_781     0.007218  0.007218

Top 5 Negative molecule structures (Decrease AUC/Sensitivity):
    Molecule_Structure  Coefficient  Abs_Coef
380            Bit_380    -0.018682  0.018682
823            Bit_823    -0.015665  0.015665
764            Bit_764    -0.014251  0.014251
158            Bit_158    -0.012704  0.012704
797            Bit_797    -0.012062  0.012062


## LN_IC50

wichtig: im folgenden steht increased AUC resistance, aber es ist IC50!

In [7]:
genomic_baseline_model(df, "LN_IC50")

Number of unique cell lines: 191
Starting Elastic Net Cross-Validation...

--- Elastic Net Results ---
Average R² Score: 0.0485
Average RMSE:     2.6802

Elastic Net selected 89 genes out of 978.
Top 5 Positive Biomarkers (Increase AUC/Resistance):
              Gene  Coefficient  Abs_Coef
934  KIFBP (26128)     0.053513  0.053513
450    VAPB (9217)     0.042209  0.042209
887    MUC1 (4582)     0.040832  0.040832
587    NOL3 (8996)     0.035844  0.035844
284    RHEB (6009)     0.035805  0.035805

Top 5 Negative Biomarkers (Decrease AUC/Sensitivity):
               Gene  Coefficient  Abs_Coef
304      BLMH (642)    -0.044524  0.044524
359     PCCB (5096)    -0.038548  0.038548
557     RPS6 (6194)    -0.036616  0.036616
422  SMNDC1 (10285)    -0.027809  0.027809
158   MAP3K4 (4216)    -0.024704  0.024704


In [10]:
chemical_baseline_model(df, 'LN_IC50')

Number of unique drugs: 295
Starting Elastic Net Cross-Validation...

--- Elastic Net Results ---
Average R² Score: -0.1571
Average RMSE:     2.9422

Elastic Net selected 209 molecule structures out of 1032.
Top 5 Positive molecule structures (Increase LN_IC50):
    Molecule_Structure  Coefficient  Abs_Coef
865            Bit_865     0.359548  0.359548
229            Bit_229     0.268557  0.268557
39              Bit_39     0.247317  0.247317
861            Bit_861     0.228568  0.228568
818            Bit_818     0.158667  0.158667

Top 5 Negative molecule structures (Decrease LN_IC50):
    Molecule_Structure  Coefficient  Abs_Coef
101            Bit_101    -0.365581  0.365581
797            Bit_797    -0.322255  0.322255
316            Bit_316    -0.310780  0.310780
297            Bit_297    -0.291528  0.291528
47              Bit_47    -0.277671  0.277671
